# Week 2 Practical: Predict Aqueous Solubility with QSAR
**AI for Drug Discovery**

In this practical, you will:
1. Load the Delaney solubility dataset
2. Generate molecular fingerprints (Morgan/ECFP)
3. Calculate physicochemical descriptors
4. Train Random Forest and XGBoost models
5. Compare model performance

In [ ]:
# Install dependencies
!pip install rdkit-pypi scikit-learn xgboost pandas matplotlib seaborn -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb

sns.set_style('whitegrid')
print('All imports successful!')

## 1. Load the Delaney Solubility Dataset
The Delaney dataset (2004) contains ~1,128 molecules with measured aqueous solubility (logS). This is a classic QSAR benchmark.

**Reference:** Delaney, J.S. (2004). ESOL: Estimating Aqueous Solubility Directly from Molecular Structure. J. Chem. Inf. Comput. Sci. 44:1000-1005

In [ ]:
# Load dataset
try:
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
    df = pd.read_csv(url)
except:
    # Fallback URL
    url = 'https://raw.githubusercontent.com/PatWalters/datafiles/main/delaney.csv'
    df = pd.read_csv(url)

print(f'Dataset: {len(df)} molecules')
print(f'Columns: {list(df.columns)}')

# Identify the SMILES and target columns
smiles_col = [c for c in df.columns if 'smiles' in c.lower()][0]
target_col = [c for c in df.columns if 'solubility' in c.lower() or 'log' in c.lower()][0]
print(f'SMILES column: {smiles_col}')
print(f'Target column: {target_col}')
df.head()

In [ ]:
# Parse SMILES
df['mol'] = df[smiles_col].apply(lambda s: Chem.MolFromSmiles(s))
valid = df['mol'].notna()
print(f'Valid molecules: {valid.sum()} / {len(df)}')
df = df[valid].reset_index(drop=True)

# Target variable
y = df[target_col].values
print(f'Target (logS) range: [{y.min():.2f}, {y.max():.2f}]')
print(f'Target mean: {y.mean():.2f}, std: {y.std():.2f}')

## 2. Generate Molecular Fingerprints (Morgan/ECFP4)
We'll use Morgan fingerprints with radius=2 (equivalent to ECFP4) and 2048 bits.

**Reference:** Rogers, D. & Hahn, M. (2010). Extended-Connectivity Fingerprints. J. Chem. Inf. Model. 50:742-754

In [ ]:
# Generate Morgan fingerprints (ECFP4)
def mol_to_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.int8)
    AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# Generate fingerprints for all molecules
fp_array = np.array([mol_to_fp(m) for m in df['mol']])
print(f'Fingerprint matrix shape: {fp_array.shape}')
print(f'Average bits set per molecule: {fp_array.sum(axis=1).mean():.1f}')

## 3. Calculate Physicochemical Descriptors

In [ ]:
# Calculate molecular descriptors
def calc_descriptors(mol):
    return {
        'MW': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'HBD': Descriptors.NumHDonors(mol),
        'HBA': Descriptors.NumHAcceptors(mol),
        'TPSA': Descriptors.TPSA(mol),
        'RotBonds': Descriptors.NumRotatableBonds(mol),
        'AromaticRings': Descriptors.NumAromaticRings(mol),
        'HeavyAtoms': Descriptors.HeavyAtomCount(mol),
        'RingCount': Descriptors.RingCount(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol),
    }

desc_df = pd.DataFrame([calc_descriptors(m) for m in df['mol']])
print(f'Descriptor matrix shape: {desc_df.shape}')
desc_df.describe().round(2)

In [ ]:
# Combine fingerprints + descriptors
X_fp = fp_array  # Fingerprints only
X_desc = desc_df.values  # Descriptors only
X_combined = np.hstack([fp_array, desc_df.values])  # Both

print(f'Fingerprints only: {X_fp.shape}')
print(f'Descriptors only: {X_desc.shape}')
print(f'Combined: {X_combined.shape}')

## 4. Train/Test Split and Model Training

In [ ]:
# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)
print(f'Training set: {X_train.shape[0]} molecules')
print(f'Test set: {X_test.shape[0]} molecules')

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=500, max_features='sqrt',
                           min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print('Random Forest Results:')
print(f'  RMSE: {rmse_rf:.3f} logS units')
print(f'  MAE:  {mae_rf:.3f}')
print(f'  R2:   {r2_rf:.3f}')

In [ ]:
# Train XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print('XGBoost Results:')
print(f'  RMSE: {rmse_xgb:.3f} logS units')
print(f'  MAE:  {mae_xgb:.3f}')
print(f'  R2:   {r2_xgb:.3f}')

## 5. Compare Models

In [ ]:
# Comparison table
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'RMSE': [rmse_rf, rmse_xgb],
    'MAE': [mae_rf, mae_xgb],
    'R2': [r2_rf, r2_xgb]
})
print(results.to_string(index=False))

In [ ]:
# Scatter plots: predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, name in zip(axes, [y_pred_rf, y_pred_xgb], ['Random Forest', 'XGBoost']):
    ax.scatter(y_test, y_pred, alpha=0.5, s=30, color='#1f77b4')
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
    ax.set_xlabel('Actual logS', fontsize=12)
    ax.set_ylabel('Predicted logS', fontsize=12)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    ax.set_title(f'{name}\nRMSE={rmse:.3f}, R2={r2:.3f}', fontsize=13)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Top 20 most important features from Random Forest
importances = rf.feature_importances_

# Create feature names
fp_names = [f'FP_bit_{i}' for i in range(2048)]
desc_names = list(desc_df.columns)
feature_names = fp_names + desc_names

# Sort by importance
idx = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(12, 6))
plt.bar(range(20), importances[idx], color='#1f77b4')
plt.xticks(range(20), [feature_names[i] for i in idx], rotation=45, ha='right')
plt.ylabel('Feature Importance')
plt.title('Top 20 Features (Random Forest)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Exercises

1. **Fingerprints only vs. descriptors only**: Train models using only fingerprints and only descriptors. Which performs better?
2. **Hyperparameter tuning**: Try different `n_estimators`, `max_depth` for XGBoost. Can you improve the RMSE?
3. **Different fingerprint radius**: Try radius=1 (ECFP2) and radius=3 (ECFP6). Does it matter?
4. **Challenge**: Implement a simple linear regression baseline. How does it compare to RF/XGBoost?

## References
- Delaney, J.S. (2004). ESOL: Estimating Aqueous Solubility Directly from Molecular Structure. JCICS 44:1000-1005
- Rogers, D. & Hahn, M. (2010). Extended-Connectivity Fingerprints. JCIM 50:742-754
- Breiman, L. (2001). Random Forests. Machine Learning 45:5-32
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. KDD 2016